# La Radiazione Cosmica di Fondo (CMB) — Planck SMICA

La **CMB** (Cosmic Microwave Background) è la luce più antica dell'universo:
fu emessa **380.000 anni dopo il Big Bang**, quando il cosmo si raffreddò
abbastanza (~3000 K) da permettere agli elettroni di legarsi ai protoni
(*ricombinazione*), rendendo l'universo trasparente. Da allora quella luce
ha viaggiato per **13.8 miliardi di anni**, stirata dall'espansione fino alle
microonde: oggi la vediamo come un bagliore a **2.725 K** che riempie tutto il
cielo.

Le sue minuscole fluttuazioni di temperatura (**~1 parte su 100.000**) sono i
**semi** da cui sono nate galassie e ammassi. In questo notebook usiamo la mappa
reale del satellite **Planck** (metodo SMICA, release PR3) per:

1. **visualizzare** la mappa di temperatura su tutto il cielo (proiezione Mollweide);
2. **mascherare** la Via Lattea (che contamina i dati);
3. studiare la **distribuzione** delle fluttuazioni (è gaussiana!);
4. calcolare lo **spettro di potenza angolare** $C_\ell$ — il grafico con i
   **picchi acustici** da cui si misurano i parametri dell'universo.

> **Nota ambiente:** questo notebook gira in locale (VSCode). Seleziona il
> kernel **"Python (CMB venv)"**. Il file dati Planck (~2 GB) viene scaricato
> dalla prima cella se non presente.

## 0. Setup e import

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os, urllib.request

print('healpy', hp.__version__)

## 1. Scarica la mappa Planck (con verifica d'integrità)

Il file SMICA a piena risoluzione (NSIDE=2048, ~50 milioni di pixel) pesa
**~2 GB**. La cella lo scarica solo se manca o se è **incompleto/troncato**
(un download interrotto è la causa più comune di errori con questi dati).

In [ ]:
DATA_DIR = "Data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = os.path.join(DATA_DIR, "COM_CMB_IQU-smica_2048_R3.00_full.fits")
URL = ("https://irsa.ipac.caltech.edu/data/Planck/release_3/all-sky-maps/"
       "maps/component-maps/cmb/COM_CMB_IQU-smica_2048_R3.00_full.fits")
EXPECTED_ROWS = 50331648   # = 12 * 2048^2 pixel HEALPix

def file_ok(path):
    """Integro se il FITS si apre e ha il numero giusto di pixel.
    Più robusto del confronto sui byte (un file valido può variare di poco)."""
    if not os.path.exists(path) or os.path.getsize(path) < 1_900_000_000:
        return False
    try:
        with fits.open(path) as h:
            return h[1].header['NAXIS2'] == EXPECTED_ROWS
    except Exception:
        return False

if file_ok(filename):
    print("File già presente e integro.")
else:
    print("File mancante o incompleto: lo scarico da IRSA (~2 GB, alcuni minuti)...")
    # scarico su file temporaneo, poi rinomino solo se integro (evita file corrotti)
    tmp = filename + ".part"
    urllib.request.urlretrieve(URL, tmp)
    os.replace(tmp, filename)
    assert file_ok(filename), "Download non valido: riesegui la cella."
    print("Download completato e verificato.")

## 2. Ispeziona il file FITS

La mappa è in formato **HEALPix**: il cielo è diviso in pixel di uguale area.
Il file contiene 10 colonne — a noi servono la temperatura `I_STOKES` (campo 0)
e la maschera di confidenza `TMASK` (campo 3).

In [ ]:
with fits.open(filename) as hdul:
    hdul.info()
    hdr = hdul[1].header
print("\nNSIDE =", hdr['NSIDE'], "| ORDERING =", hdr['ORDERING'],
      "| unità =", hdr['TUNIT1'])
print("Numero pixel attesi:", 12 * hdr['NSIDE']**2)

## 3. Carica la mappa di temperatura

Il file usa ordinamento **NESTED** (lo dice l'header). Con `nest=True` la
leggiamo correttamente. `hp.read_map` restituisce un array con un valore di
temperatura (in K) per ogni pixel del cielo.

In [ ]:
cmb_T = hp.read_map(filename, field=0, nest=True)   # I_STOKES, in K_CMB
nside = hp.get_nside(cmb_T)
print(f"Mappa caricata: {len(cmb_T):,} pixel, NSIDE={nside}")
print(f"Temperatura: media={np.mean(cmb_T):.2e} K, "
      f"fluttuazione tipica (std)={np.std(cmb_T)*1e6:.1f} µK")

## 4. La mappa del cielo (Mollweide)

La proiezione **Mollweide** mostra l'intera sfera celeste su un ovale. I colori
sono le fluttuazioni di temperatura rispetto alla media: **±200 µK**
(0.0002 K). Rosso = punti leggermente più caldi, blu = più freddi.

La banda rossa orizzontale al centro **NON è CMB**: è l'emissione della nostra
**Via Lattea**, che qui contamina il segnale. La rimuoveremo al passo 5.

In [ ]:
hp.mollview(cmb_T, nest=True,
            title="Planck PR3 — Temperatura CMB (SMICA)",
            unit="K", cmap="coolwarm", min=-2e-4, max=2e-4)
hp.graticule()
plt.show()

## 5. Applica la maschera galattica

`TMASK` vale 1 dove i dati sono affidabili e 0 dove la Via Lattea (o sorgenti
puntiformi) contamina troppo. Mettiamo a `hp.UNSEEN` (valore "trasparente" per
healpy) i pixel esclusi. Registriamo anche la **frazione di cielo** usata
(`fsky`): servirà a correggere lo spettro di potenza.

In [ ]:
tmask = hp.read_map(filename, field=3, nest=True)    # TMASK
mask_bool = tmask > 0.5

cmb_masked = cmb_T.copy()
cmb_masked[~mask_bool] = hp.UNSEEN

fsky = mask_bool.sum() / len(mask_bool)
print(f"Frazione di cielo mantenuta (fsky): {fsky*100:.1f}%")

hp.mollview(cmb_masked, nest=True,
            title=f"CMB mascherata (fsky={fsky*100:.0f}%)",
            unit="K", cmap="coolwarm", min=-2e-4, max=2e-4)
hp.graticule()
plt.show()

## 6. Distribuzione delle fluttuazioni

Un fatto profondo: le fluttuazioni della CMB sono **quasi perfettamente
gaussiane**. Questa è una previsione chiave dell'**inflazione cosmica** (le
fluttuazioni nascono da fluttuazioni quantistiche amplificate). Confrontiamo
l'istogramma dei dati reali con una gaussiana della stessa deviazione standard.

In [ ]:
valid = cmb_masked[cmb_masked != hp.UNSEEN]
sigma = np.std(valid)

plt.figure(figsize=(8,5))
plt.hist(valid*1e6, bins=120, density=True, alpha=0.7,
         color="#3f51b5", label="Dati Planck")
x = np.linspace(valid.min(), valid.max(), 400)
gauss = np.exp(-0.5*(x/sigma)**2) / (sigma*np.sqrt(2*np.pi))
plt.plot(x*1e6, gauss/1e6, 'r-', lw=2, label=f"Gaussiana (σ={sigma*1e6:.0f} µK)")
plt.xlabel("ΔT [µK]"); plt.ylabel("PDF")
plt.title("Distribuzione delle fluttuazioni CMB — è gaussiana")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 7. Isotropia: emisfero Nord vs Sud

L'universo su grande scala dovrebbe apparire **uguale in ogni direzione**
(isotropia). Verifichiamolo confrontando la deviazione standard delle
fluttuazioni nell'emisfero galattico Nord e Sud. Dovrebbero essere molto simili
(la piccola differenza osservata da Planck è la dibattuta "anomalia di
asimmetria").

In [ ]:
npix = len(cmb_masked)
theta, phi = hp.pix2ang(nside, np.arange(npix), nest=True)
lat = np.pi/2 - theta   # latitudine: >0 Nord, <0 Sud

seen = cmb_masked != hp.UNSEEN
north = (lat > 0) & seen
south = (lat < 0) & seen

print(f"Std Nord:  {np.std(cmb_masked[north])*1e6:.2f} µK  ({north.sum():,} pixel)")
print(f"Std Sud:   {np.std(cmb_masked[south])*1e6:.2f} µK  ({south.sum():,} pixel)")
print(f"Differenza relativa: {abs(np.std(cmb_masked[north])-np.std(cmb_masked[south]))/np.std(valid)*100:.1f}%")

## 8. Lo spettro di potenza angolare $C_\ell$

È **il** grafico della cosmologia moderna. Scomponiamo la mappa in armoniche
sferiche: $C_\ell$ misura quanta "potenza" (varianza) c'è a ogni scala angolare,
indicizzata dal **multipolo** $\ell$ (ℓ grande = scale piccole).

**Due accorgimenti fondamentali** (che il codice ingenuo sbaglia):
1. `anafast` su una mappa mascherata richiede ordinamento **RING**, non NESTED
   → convertiamo prima.
2. La maschera **riduce la potenza** proporzionalmente a `fsky` → correggiamo
   dividendo per `fsky` (correzione di prima approssimazione, "MASTER").

Limitiamo a $\ell_{max}=2000$ per tempi ragionevoli.

In [ ]:
# anafast lavora in ordinamento RING: converti la mappa mascherata
cmb_ring = hp.reorder(cmb_masked, n2r=True)
# i pixel UNSEEN vanno azzerati per anafast (contribuiscono 0 alla trasformata)
cmb_ring = np.where(cmb_ring == hp.UNSEEN, 0.0, cmb_ring)

LMAX = 2000
cl_tt = hp.anafast(cmb_ring, lmax=LMAX)
cl_tt = cl_tt / fsky            # correzione approssimata per il cielo mascherato

ell = np.arange(len(cl_tt))
print(f"Spettro calcolato fino a ℓ={LMAX}")

In [ ]:
# D_ell = ell(ell+1)C_ell/2pi  in µK^2 (la quantità che si grafica sempre)
Dl = ell*(ell+1)*cl_tt/(2*np.pi) * 1e12   # *1e12: da K^2 a µK^2

plt.figure(figsize=(10,6))
plt.plot(ell, Dl, color="#90CAF9", lw=0.6, alpha=0.6, label="spettro grezzo")

# curva lisciata: rende visibili i picchi acustici
Dl_s = np.convolve(Dl, np.ones(21)/21, mode='same')
plt.plot(ell, Dl_s, color="#D32F2F", lw=2, label="spettro lisciato")

# marca i primi tre picchi acustici
for lo, hi in [(150,300),(400,700),(700,1000)]:
    r = (ell>=lo)&(ell<=hi); lpk = ell[r][np.argmax(Dl_s[r])]
    plt.axvline(lpk, color="#555", ls=":", lw=1)
    plt.annotate(f"ℓ≈{lpk}", (lpk, Dl_s[lpk]), textcoords="offset points",
                 xytext=(5,8), fontsize=9, color="#333")

plt.xlim(2, LMAX); plt.ylim(0, None)
plt.xlabel(r"Multipolo $\ell$", fontsize=12)
plt.ylabel(r"$\mathcal{D}_\ell=\ell(\ell+1)C_\ell/2\pi\ \ [\mu K^2]$", fontsize=12)
plt.title("Planck PR3 — Spettro di potenza CMB TT (SMICA)", fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 9. Cosa dicono i picchi acustici?

La curva sale fino a un **primo picco intorno a ℓ ≈ 220** (scala angolare ~1°),
poi oscilla con picchi successivi. Sono le **oscillazioni acustiche
barioniche**: onde sonore nel plasma primordiale, "congelate" al momento della
ricombinazione.

Da questi picchi si misurano i parametri fondamentali dell'universo:
- **posizione del 1° picco** → l'universo è **spazialmente piatto**;
- **altezze relative dei picchi** → quanta **materia ordinaria** vs **materia
  oscura**;
- lo **smorzamento** ad alti ℓ → la fisica del disaccoppiamento.

È così che Planck ha misurato che l'universo è fatto per **~5% di materia
ordinaria, ~27% di materia oscura, ~68% di energia oscura**.

*(Nota: questo spettro "fatto in casa" da una singola mappa è più rumoroso di
quello ufficiale Planck, che combina più stime e una deconvoluzione completa
della maschera. Ma i picchi acustici si vedono comunque — ed è già straordinario
ricavarli da sé.)*

In [ ]:
# Cerca i primi tre picchi acustici su uno spettro LEGGERMENTE lisciato
# (lo spettro grezzo è troppo rumoroso: argmax puntuale sbaglierebbe)
Dl_smooth = np.convolve(Dl, np.ones(21)/21, mode='same')

print("Picchi acustici stimati da questa mappa:")
for lo, hi, atteso in [(150,300,220), (400,700,540), (700,1000,810)]:
    r = (ell >= lo) & (ell <= hi)
    lpk = ell[r][np.argmax(Dl_smooth[r])]
    print(f"  ℓ ≈ {lpk:4d}   (atteso Planck ≈ {atteso})")

# Salva lo spettro per usi successivi
np.save(os.path.join(DATA_DIR, "cl_tt_smica.npy"), cl_tt)
print("\nSpettro salvato in Data/cl_tt_smica.npy")

---
# 10. Le Anomalie della CMB

Finora abbiamo visto il segnale **atteso** (fluttuazioni gaussiane, picchi
acustici) che conferma il modello standard e l'inflazione. Ora guardiamo le
**anomalie**: caratteristiche che il modello **non** prevede facilmente e che
restano dibattute. Potrebbero essere indizi di **nuova fisica**, oppure semplici
casualità statistiche (abbiamo un solo universo da osservare → *varianza
cosmica*). Ne esploriamo tre, tutte con i dati reali di Planck.

> ⚠️ **Cautela scientifica:** queste anomalie sono *reali nei dati* ma la loro
> *interpretazione* è aperta. Non sono prove di nuova fisica — sono domande
> senza risposta.

---
# 10. Le Anomalie della CMB

Finora abbiamo visto il segnale **atteso** (fluttuazioni gaussiane, picchi
acustici) che conferma il modello standard e l'inflazione. Ora guardiamo le
**anomalie**: caratteristiche che il modello **non** prevede facilmente e che
restano dibattute. Potrebbero essere indizi di **nuova fisica**, oppure semplici
casualità statistiche (abbiamo un solo universo da osservare → *varianza
cosmica*). Ne esploriamo tre, tutte con i dati reali di Planck.

> ⚠️ **Cautela scientifica:** queste anomalie sono *reali nei dati* ma la loro
> *interpretazione* è aperta. Non sono prove di nuova fisica — sono domande
> senza risposta.

## 10.1 Preparazione: mappa in ordinamento RING

Le analisi che seguono (dischi sul cielo, armoniche sferiche) sono più comode in
ordinamento **RING**. Convertiamo una volta e riusiamo.

In [ ]:
# Mappa e maschera in RING (le celle sopra lavorano in NESTED)
cmb_ring_full = hp.reorder(cmb_T, n2r=True)
mask_ring = hp.reorder(mask_bool.astype(float), n2r=True) > 0.5
print("Convertite in RING. Pixel validi:", mask_ring.sum())

## 10.2 Il Cold Spot 🔵

Nell'emisfero sud c'è una regione **anomalamente fredda e grande**, centrata
attorno alle coordinate galattiche **l ≈ 209°, b ≈ −57°**. Una macchia fredda
così estesa (~5–10°) ha una probabilità piccola di comparire per caso nel modello
standard. Le ipotesi vanno da un gigantesco **vuoto cosmico** lungo la linea di
vista, a effetti più esotici. Misuriamone la temperatura media.

In [ ]:
# Coordinate galattiche note del Cold Spot
l_cs, b_cs = 209.0, -57.0
vec_cs = hp.ang2vec(np.radians(90 - b_cs), np.radians(l_cs))

# pixel entro 5 gradi dal centro
disc = hp.query_disc(nside, vec_cs, np.radians(5))
T_coldspot = np.mean(cmb_ring_full[disc]) * 1e6      # µK
T_sky      = np.mean(cmb_ring_full[mask_ring]) * 1e6  # µK

print(f"Temperatura media nel Cold Spot (5°): {T_coldspot:+.1f} µK")
print(f"Temperatura media del cielo:          {T_sky:+.1f} µK")
print(f"=> Il Cold Spot è più FREDDO di ~{abs(T_coldspot - T_sky):.0f} µK")

# Visualizza uno zoom (gnomview) centrato sul Cold Spot
hp.gnomview(cmb_ring_full, rot=(l_cs, b_cs), reso=8, xsize=200,
            title="Cold Spot (zoom)", unit="K", cmap="coolwarm",
            min=-2e-4, max=2e-4)
plt.show()

## 10.3 Asimmetria emisferica ↕️

Il modello standard dice che l'universo è **isotropo**: le fluttuazioni devono
avere la stessa "forza" (varianza) in ogni direzione. Ma Planck (e prima WMAP)
trova che un emisfero ha fluttuazioni sistematicamente **più forti** dell'altro.

Non ci limitiamo a Nord/Sud galattico (già visto): cerchiamo la **direzione di
massima asimmetria**, dividendo il cielo in molte coppie di emisferi opposti e
confrontando la varianza.

In [ ]:
# Degrada la mappa (più veloce) e scandaglia molte direzioni
nside_lo = 64
cmb_lo = hp.ud_grade(np.where(mask_ring, cmb_ring_full, hp.UNSEEN),
                     nside_lo, order_in='RING')
seen_lo = cmb_lo != hp.UNSEEN

npix_lo = hp.nside2npix(nside_lo)
vecs = np.array(hp.pix2vec(nside_lo, np.arange(npix_lo))).T  # direzione di ogni pixel

# prova ~200 assi di divisione; per ciascuno confronta varianza dei due emisferi
axes_idx = np.arange(0, npix_lo, max(1, npix_lo // 200))
best = None
for i in axes_idx:
    axis = vecs[i]
    dot = vecs @ axis                     # >0 emisfero "positivo", <0 opposto
    hemN = seen_lo & (dot > 0)
    hemS = seen_lo & (dot < 0)
    if hemN.sum() < 50 or hemS.sum() < 50:
        continue
    vN, vS = np.var(cmb_lo[hemN]), np.var(cmb_lo[hemS])
    asym = abs(vN - vS) / (vN + vS)       # contrasto di varianza
    if best is None or asym > best[0]:
        best = (asym, i, vN, vS)

asym, ipix, vN, vS = best
l_ax, b_ax = hp.pix2ang(nside_lo, ipix, lonlat=True)
print(f"Massima asimmetria trovata lungo l≈{l_ax:.0f}°, b≈{b_ax:.0f}°")
print(f"  varianza emisfero forte: {vN*1e12:.0f} µK²   debole: {vS*1e12:.0f} µK²")
print(f"  contrasto: {asym*100:.1f}%")
print("(Planck riporta l'asse dell'asimmetria attorno a l~220°, b~-20°)")

## 10.4 I bassi multipoli: quadrupolo e ottupolo 🌀

Le strutture a **grandissima scala** (multipoli ℓ=2 "quadrupolo" e ℓ=3
"ottupolo") mostrano due stranezze:
1. il **quadrupolo è più debole** del previsto;
2. quadrupolo e ottupolo appaiono **allineati** tra loro e con il piano
   dell'eclittica/sistema solare — soprannominato *"axis of evil"*. Non dovrebbe
   esserci alcuna relazione col nostro sistema solare, quindi l'allineamento è
   sospetto.

Ricostruiamo la mappa dei **soli** ℓ=2 e ℓ=3 per vederne la forma.

In [ ]:
# estrai solo i multipoli bassi (ell = 2 e 3)
cmb_zeroed = np.where(mask_ring, cmb_ring_full, 0.0)
alm = hp.map2alm(cmb_zeroed, lmax=3)
# tieni solo ell=2,3 (azzera monopolo ell=0 e dipolo ell=1)
for ell_rm in [0, 1]:
    for m in range(ell_rm + 1):
        alm[hp.Alm.getidx(3, ell_rm, m)] = 0.0
low_map = hp.alm2map(alm, nside=128, lmax=3)

hp.mollview(low_map, title="Quadrupolo + Ottupolo (ℓ=2,3) — 'axis of evil'?",
            unit="K", cmap="coolwarm")
hp.graticule()
plt.show()
print("Questa è la struttura più grande della CMB: solo 4 zone calde/fredde.")
print("La loro orientazione stranamente allineata è una delle anomalie aperte.")

## 10.5 Riepilogo delle anomalie

| Anomalia | Cosa si osserva | Interpretazione |
|---|---|---|
| **Cold Spot** | macchia fredda troppo grande (~-100 µK) a l=209°,b=-57° | vuoto cosmico? nuova fisica? caso? |
| **Asimmetria emisferica** | un emisfero fluttua più dell'altro | violazione dell'isotropia? varianza cosmica? |
| **Bassi multipoli** | quadrupolo debole + allineamento "axis of evil" | coincidenza? contaminazione? fisica esotica? |

**Il punto chiave:** nessuna di queste è, oggi, spiegata con certezza. Sono le
domande aperte lasciate dalla mappa più precisa mai fatta dell'universo neonato.
Il modello standard (inflazione + ΛCDM) spiega *magnificamente* il 99% dei dati
— queste anomalie sono il residuo che potrebbe (o no) nascondere qualcosa di
nuovo.